In [ ]:
!pip -q install langchain langchain_community openai tiktoken sentence-transformers==2.2.2 requests==2.32.4 faiss-cpu pypdf python-docx ragas anthropic langchain-anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
   ━━

In [ ]:
import os
from ragas import RunConfig

os.environ["RAGAS_DEBUG"] = "true"
run_config = RunConfig(timeout=120, log_tenacity=True)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings

# Supply OPENAI_API_KEY through the environment; never save credentials in this notebook.

In [ ]:
vectorstorepath = f"work/indexes/worringer_index"
answerpath = f"GPT_4o_extractions_full_pipe/worringer_extraction_4o"
groundtruthpath = f"Annotations/worringer.docx"

In [ ]:
openai_embeddings = OpenAIEmbeddings()

# Load FAISS
KB = FAISS.load_local(
    vectorstorepath,
    openai_embeddings,
    allow_dangerous_deserialization=True
)
chunks = KB.index.ntotal
#chunks = chunks/1
#chunks=62
retriever = KB.as_retriever(search_kwargs={"k": chunks})
print(f"Number of Chunks in VDB: {chunks}")

/tmp/ipykernel_1800/3468371221.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  openai_embeddings = OpenAIEmbeddings()


Number of Chunks in VDB: 10


In [ ]:
import re
from docx import Document

doc = Document(answerpath)
full_text = []

# paragraphs
for para in doc.paragraphs:
    full_text.append(para.text)

# tables
for table in doc.tables:
    for row in table.rows:
        cells = [cell.text.strip() for cell in row.cells]
        full_text.append(" | ".join(cells))
        full_text.append("")

answer = "\n".join(full_text)
answer = re.sub(r'[*,-]', '', answer)

In [ ]:
doc = Document(groundtruthpath)
gt_text = []

for para in doc.paragraphs:
    gt_text.append(para.text)

for table in doc.tables:
    for row in table.rows:
        cells = [cell.text.strip() for cell in row.cells]
        gt_text.append(" | ".join(cells))
        gt_text.append("")

groundtruth = "\n".join(gt_text)

In [ ]:
query = '''Provide a comprehensive structured summary of the Women’s Health Initiative (WHI) randomized controlled trial evaluating hormone therapy in postmenopausal women.

Include the following required fields:

Study Methods

Study design

Allocation concealment

Blinding

Setting

Sample size

Duration of follow-up

Whether intention-to-treat analysis was performed

Population

Inclusion criteria

Key exclusion criteria

Baseline characteristics (mean age, sex distribution, race/ethnicity)

Intervention

Hormone therapy regimen(s), dosage, and group size

Comparator

Description of placebo or control group

Outcomes

Primary outcomes

Secondary outcomes

Adverse events

Main Results (Statistical Data Required)
For each outcome, report:

Outcome type

Outcome description

Time point or subgroup

Effect size (HR or RR) with 95% confidence interval

Comparison interpretation

Sources of Funding

Overall Conclusion and Clinical Recommendation'''

In [ ]:
retrieved_docs = retriever.get_relevant_documents(query)
context = [doc.page_content for doc in retrieved_docs]

/tmp/ipykernel_1800/3072982058.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)


In [ ]:
print(context[0])

In [ ]:
from datasets import Dataset
data = {
    "question": [query],
    "answer": [answer],
    "contexts": [context],
    "ground_truth": [groundtruth]
}
dataset1 = Dataset.from_dict(data)

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    context_recall,
    #answer_relevancy
)

import nest_asyncio
nest_asyncio.apply()

/tmp/ipykernel_1800/367557245.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1800/367557245.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (


In [ ]:
from langchain_anthropic import ChatAnthropic
# Supply ANTHROPIC_API_KEY through the environment; never save credentials in this notebook.

claude_3_haiku = ChatAnthropic(
    model="claude-3-haiku-20240307",
    max_tokens=4096,
    temperature=0.2
)

In [ ]:
from langchain.chat_models import ChatOpenAI
gpt_5_1 = ChatOpenAI(model_name='gpt-5.1')

/tmp/ipykernel_1800/118763364.py:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  gpt_5_1 = ChatOpenAI(model_name='gpt-4.1-mini')


In [ ]:
try:
    result = evaluate(
        llm=gpt_5_1,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness,context_recall],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
df #worringer

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Research Original Investigation\nAcetaminophe...,PICO variables\n1. Methods\n Design: Nation...,0.903846


In [ ]:
df #wilken

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Articles\nResearch in context\nEvidence befor...,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df #visva

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[USPSTF Recommendation: Screening for Breast C...,PICO variables\nThis response is based on the ...,1.0


In [ ]:
df #tanner

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Research Original Investigation\nOpt-Out vs O...,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df #parks

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[RESEARCH\n.\nat Birmingham City University\n ...,PICO variables\n1. Methods\n Design: Nation...,0.983607


In [ ]:
df #ott

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Clinical Review & Education US Preventive Ser...,PICO variables\n1. Methods\n Design: Random...,0.977011


In [ ]:
df #maki

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[USPSTF Recommendation: Screening for Asymptom...,PICO variables\nThe provided text is a summary...,1.0


In [ ]:
df #kabo

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Articles\n187 473 women invited\n171 063 no r...,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df #haung

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Articles\npost-baseline visit. Baseline demog...,PICO variables\n1. Methods\n Design: Random...,0.914286


In [ ]:
df #buden

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[USPSTF Recommendation: Screening for Preeclam...,PICO variables\nThe information provided does ...,1.0


In [ ]:
df #batur

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Self-Sampling Strategy to Increase Cervical C...,PICO variables\n1. Methods\n Design: Random...,0.966667


In [ ]:
df # batur 2025

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[RESEARCH\nRESEARCH\n.\nat Birmingham City Uni...,PICO variables\n1. Methods\n Design: Nation...,0.986111


In [ ]:
df #batur 2025 4

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[RESEARCH\nRESEARCH\n.\nat Birmingham City Uni...,PICO variables\n1. Methods\n Design: Nation...,NaN


In [ ]:
df #batur 2023

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,"[Articles\ntest was carried out, and a positiv...",PICO variables\n1. Methods\n Design: Random...,0.90991


In [ ]:
df #batur 2021

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Articles\nscreening in a 1:1:2 ratio. The tri...,PICO variables\n1. Methods\n Design: Random...,0.977778


In [ ]:
df #12th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Cochrane\nLibrary\nTrusted evidence.\nInforme...,PICO variables\nCertainly! Below is a structur...,1.0


In [ ]:
df #11th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Exemestane to Prevent Breast Cancer\ntectomy....,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df #10th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[BMJ 2012;345:e6409 doi: 10.1136/bmj.e6409 (Pu...,PICO variables\n1. Methods\n Design: Random...,0.983051


In [ ]:
df #9th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,"[14710528, 2017, 10, Downloaded from https://o...",PICO variables\n1. Methods\n Design: System...,0.962963


In [ ]:
df #8th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Chlebowski et al.\nPage 3\nAll women in both ...,PICO variables\n1. Methods\n Design: Random...,0.981481


In [ ]:
df #6th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Maalouf et al.\nPage 14\nIncidence (Annualize...,PICO variables\n1. Methods\n Design: Random...,0.969697


In [ ]:
df #5th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Cochrane\nLibrary\nTrusted evidence.\nInforme...,PICO variables\nTo address the question using ...,0.984375


In [ ]:
df #4th

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Manson et al.\nPage 19\nTable\nBaseline Chara...,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df # 3rd

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[McCurry et al.\nPage 5\ncontacted by telephon...,PICO variables\n1. Methods\n Design: Random...,0.931818


In [ ]:
df #3rd

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[McCurry et al.\nPage 5\ncontacted by telephon...,PICO variables\n1. Methods\n Design: Random...,1.0


In [ ]:
df #2nd

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Canonico et al Hormone Therapy and Stroke ...,PICO variables\n1. Methods\n Design: Nested...,1.0


In [ ]:
df #2nd

,user_input,retrieved_contexts,response,faithfulness
0,Provide a comprehensive structured summary of ...,[Canonico et al Hormone Therapy and Stroke ...,PICO variables\n1. Methods\n Design: Nested...,0.864198


In [ ]:
print(len(df['retrieved_contexts']))

1


In [ ]:
for x in df['retrieved_contexts']:
  print(x)

['Clinical Review & Education US Preventive Services Task Force\nUSPSTF Recommendation Statement: Hormone Therapy After Menopause\nFigure 1. US Preventive Services Task Force (USPSTF) Grades and Levels of Certainty\nWhat the USPSTF Grades Mean and Suggestions for Practice\nGrade\nDefinition\nSuggestions for Practice\nA\nThe USPSTF recommends the service. There is high certainty that the net benefit is substantial.\nOffer or provide this service.\nB\nThe USPSTF recommends the service. There is high certainty that the net benefit is moderate, or\nthere is moderate certainty that the net benefit is moderate to substantial.\nOffer or provide this service.\nC\nThe USPSTF recommends selectively offering or providing this service to individual patients\nbased on professional judgment and patient preferences. There is at least moderate certainty\nthat the net benefit is small.\nOffer or provide this service for selected\npatients depending on individual\ncircumstances.\nD\nThe USPSTF recommend

In [ ]:
try:
    result = evaluate(
        llm=claude_3_haiku,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: OutputParserException(Invalid json output: {
    "statements": [
        {
            "statement": "The study was a randomized controlled trial.",
            "reason": "The study design is clearly described as "randomized, double-blind, multinational, and placebo-controlled trials".",
            "verdict": 1
        },
        {
            "statement": "The study used a centralized interactive voice/web response system for randomization which indicates allocation concealment.",
            "reason": "The study states that "Randomization was performed centrally using an interactive voice/web response system and stratified by North America and the rest of the countries."",
            "verdict": 1
        },
        {
            "statement": "Investigators, participants, and study personnel remained blinded throughout the trials.",
            "reason": "The study explicitly states that "Investigators, participants, and study personne

In [ ]:
try:
    result = evaluate(
        llm=claude_3_haiku,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: OutputParserException(Invalid json output: {
    "statements": [
        {
            "statement": "The study design was a randomized controlled trial.",
            "reason": "The passage states that this was a \"randomized controlled study, with parallel randomization (1:1:1)\".",
            "verdict": 1
        },
        {
            "statement": "The allocation concealment was achieved by using sequentially numbered sealed opaque envelopes.",
            "reason": "The passage mentions that \"Allocation was concealed in sequentially numbered, sealed, opaque envelopes.\"",
            "verdict": 1
        },
        {
            "statement": "The therapist who carried out the evaluation and treatment was not blinded.",
            "reason": "The passage states that \"The main limitation of our study was that the therapist that carried out the evaluation and treatment was not blinded and this could have influenced the results.\"",

In [ ]:
try:
    result = evaluate(
        llm=gpt_5_1,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
df # dee

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[doi: 10.1210/jc.2019-00677\nhttps://academic....,PICO variables\n1. Methods\n Design: Random...,Design: Randomized control trial\nDrug & dose ...,1.0


In [ ]:
try:
    result = evaluate(
        llm=gpt_5_1,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
try:
    result = evaluate(
        llm=claude_3_haiku,
        dataset=dataset1,
        raise_exceptions=False,
        metrics=[faithfulness],
    )

    df = result.to_pandas()


except KeyError as e:
    print("Error during evaluation:", e)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: OutputParserException(Invalid json output: {
    "statements": [
        {
            "statement": "The study was a randomized controlled trial.",
            "reason": "The study design is clearly described as "randomized, double-blind, multinational, and placebo-controlled trials".",
            "verdict": 1
        },
        {
            "statement": "The study used a centralized interactive voice/web response system for randomization which indicates allocation concealment.",
            "reason": "The study states that "Randomization was performed centrally using an interactive voice/web response system and stratified by North America and the rest of the countries."",
            "verdict": 1
        },
        {
            "statement": "Investigators, participants, and study personnel remained blinded throughout the trials.",
            "reason": "The study explicitly states that "Investigators, participants, and study personne

In [ ]:
df # oasis

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Elinzanetant for Vasomotor Symptoms Associate...,PICO variables\n1. Methods\n Design: Randomize...,Design: Randomized control trial\nDrug & dose ...,0.977273


In [ ]:
df # oasis

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Elinzanetant for Vasomotor Symptoms Associate...,PICO variables\n1. Methods\n Design: Randomize...,Design: Randomized control trial\nDrug & dose ...,0.780303


In [ ]:
df # oasis

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Elinzanetant for Vasomotor Symptoms Associate...,PICO variables\n1. Methods\n Design: Randomize...,Design: Randomized control trial\nDrug & dose ...,0.645833


In [ ]:
df # sky

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Articles\nThe mixed model repeated measures u...,PICO variables\n1. Methods\n Design: Random...,Design: Randomized control trial\nDrug & dose ...,0.978495


In [ ]:
df # Oasis

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Elinzanetant for Vasomotor Symptoms Associate...,PICO variables\n1. Methods\n Design: Randomize...,Design: Randomized control trial\nDrug & dose ...,1.0


In [ ]:
df # iLankoon

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Page 4 of 9\nIlankoon et al. BMC Women’s Heal...,PICO variables\n1. Methods\n Design: Qualit...,"Country of Study: Colombo, Sri Lanka\nAim: To ...",1.0


In [ ]:
df # desalis

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[540\nI. DE SALIS ET AL.\nStrauss, A. (1987). ...",PICO variables\nThe provided text does not des...,Country of Study:\nUK\nAim:\nTo explore the ex...,1.0


In [ ]:
df # Wang

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[1423\nWorld Journal of Urology (2019) 37:1421...,PICO variables\n1. Methods\n Design: Random...,1. Methods\nDesign: Randomized controlled tria...,1.0


In [ ]:
df # verschoor 3

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[Journals of Gerontology: MEDICAL SCIENCES, 20...",PICO variables\n1. Methods\n Design: Crosss...,"Location: Canada\nSample Size: 15,320\nMean Ag...",1.0


In [ ]:
df # Verschoorn

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[Journals of Gerontology: MEDICAL SCIENCES, 20...",PICO variables\n1. Methods\n Design: Crosss...,"Location: Canada\nSample Size: 15,320\nMean Ag...",1.0


In [ ]:
df # Simkin

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[214\nSimkin-Silverman et al.\nAnnals of Behav...,PICO variables\n1. Methods\n Design: Random...,Country: USA\nDesign: Randomized Controlled Tr...,1.0


In [ ]:
df # Pereira

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[15206777, 2013, 1, Downloaded from https://on...",PICO variables\n1. Methods\n Design: Random...,1. Methods\nDesign: Randomized controlled tria...,1.0


In [ ]:
df # Ogwumike

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Niger. J. Physiol. Sci. 26 (2011): Ogwumike e...,PICO variables\n1. Methods\n Design: Pretes...,Country: Nigeria\nDesign: RCT\nParticipants: 2...,1.0


In [ ]:
df # Lee

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Y. Lee et al.\nArchives of Gerontology and Ge...,PICO variables\n1. Methods\n\n Design: Cros...,Location: South Korea\nSample Size: 1264\nMean...,1.0


In [ ]:
df # Leibaschoff 3.5

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,[Gynecology\nSURGICAL TECHNOLOGY INTERNATIONAL...,PICO variables\n1. Methods\n Design: Prospe...,1. Methods\nDesign: Randomized controlled tria...,1.0


In [ ]:
df # Kojima 3.5 2nd

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[15325415, 2022, 9, Downloaded from https://ag...",PICO variables\n1. Methods\n Design: Prospe...,\n\nLocation: UK\nSample size: 1249\nMean age ...,1.0


In [ ]:
df # Kojima 3

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[15325415, 2022, 9, Downloaded from https://ag...",PICO variables\n1. Methods\n Design: Prospe...,\n\nLocation: UK\nSample size: 1249\nMean age ...,1.0


In [ ]:
df # haung 3 2nd

,user_input,retrieved_contexts,response,reference,faithfulness
0,Extract structured PICO information from the R...,"[15325415, 2018, 11, Downloaded from https://a...",PICO variables\n1. Methods\n Design: Prospe...,Location: US\nSample Size: 7699\nMean Age (Ran...,1.0


In [ ]:
df # haung 3

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,"[15325415, 2018, 11, Downloaded from https://a...",PICO variables\n1. Methods\n Design: Prospe...,Location: US\nSample Size: 7699\nMean Age (Ran...,0.785281,1.0


In [ ]:
df # hanger 3

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,"[HAGNER ET AL\n[HDL], low-density lipoprotein ...",PICO variables\n1. Methods\n Design: The st...,Country: Poland\nDesign: Pre/Post\nParticipant...,0.755984,1.0


In [ ]:
df # Capobiano 3.5

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,[400\nArch Gynecol Obstet (2012) 285:397–403\n...,PICO variables\n1. Methods\n Design: Random...,\n1. Methods\nDesign: Randomized controlled tr...,0.740051,1.0


In [ ]:
df #worringer

,user_input,retrieved_contexts,response,reference,faithfulness,context_recall
0,Provide a comprehensive structured summary of ...,[Research Original Investigation\nAcetaminophe...,PICO variables\n1. Methods\n Design: Nation...,Methods\nDesign: Nationwide cohort study with ...,0.947368,0.904762


In [ ]:
df # Capobiano 3

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,[400\nArch Gynecol Obstet (2012) 285:397–403\n...,PICO variables\n1. Methods\n Design: Random...,\n1. Methods\nDesign: Randomized controlled tr...,0.855731,1.0


In [ ]:
df # Bertotto 3 haiku 2nd time

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,"[15206777, 2017, 8, Downloaded from https://on...",PICO variables\n1. Methods\n Design: Random...,1. Methods\nDesign: Randomized controlled tria...,0.822708,0.947368


In [ ]:
df # Bertotto 3.5

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,"[15206777, 2017, 8, Downloaded from https://on...",PICO variables\n1. Methods\n Design: Random...,1. Methods\nDesign: Randomized controlled tria...,0.736125,1.0


In [ ]:
df # Bertotto

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness
0,Extract structured PICO information from the R...,"[15206777, 2017, 8, Downloaded from https://on...",PICO variables\n1. Methods\n Design: Random...,1. Methods\nDesign: Randomized controlled tria...,0.853318,0.95
